# Validation, second AOI (transfer test)

Copy of `04_validation.ipynb` for a second 10 × 10 km AOI. The centre is set in the first code cell, so `layer_config.py` (main AOI) is unchanged. All outputs go to `outputs/validation_aoi2/`.

Run order: first run the export cell in `04_validation.ipynb` (main AOI) to create `outputs/validation/rf_training_samples_aoi1.csv`, then Run All here.

## 0. Load config (second AOI) and initialise Earth Engine

In [ ]:
import sys, importlib, io, zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg
importlib.reload(cfg)

# --- SECOND AOI (transfer test): override the centre here, layer_config.py is left unchanged ---
cfg.MAP_CENTER = [-39.86, 176.70]
cfg.AOI_BOUNDS = cfg.aoi_bounds()

_repo = _root if (_root / "processing" / "ee_auth.py").exists() else _root.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from processing.ee_auth import initialize_ee

initialize_ee()

aoi = cfg.aoi_geometry()
SCALE = cfg.WORKING_SCALE
AREA_SCALE = 20                      # same as 01_ section 5b

RAW_DIR = _repo / "data" / "raw" / "validation"
OUT_DIR = _repo / "outputs" / "validation_aoi2"   # separate folder: main-AOI results are not overwritten
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

FLOOD_REF_URL = "https://files.dragonfly.co.nz/data/hawkes-bay-flood/hawkes-bay-flood-2023-02-14.zip"
IMPACT_REF_URL = "https://files.dragonfly.co.nz/data/hawkes-bay-flood/hawkes-bay-flood-silt-slip-2023-02-21.zip"

print("AOI:", cfg.MAP_CENTER, f"±{cfg.AOI_RADIUS_KM} km |", cfg.START_DATE, "→", cfg.END_DATE)

## 1. Sentinel-1 event and historical stacks
Same as `01_` sections 1, 1b, 2.

In [ ]:
def s1_iw_vv_vh_angle(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .select(["VV", "VH", "angle"])
    )

# Ancillary layers
_fabdem = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM").filterBounds(aoi)
dem = _fabdem.mosaic().select("b1").setDefaultProjection(_fabdem.first().projection()).clip(aoi).rename("elevation")
slope_deg = ee.Terrain.slope(dem).rename("slope")
worldcover = ee.Image("ESA/WorldCover/v200/2021").clip(aoi)
wc_water = worldcover.eq(80)
dark_land = ee.Image(0)
for _cls in cfg.EXCLUDE_WORLDCOVER_CLASSES:
    dark_land = dark_land.Or(worldcover.eq(_cls))
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi)
gsw_permanent = gsw.select("occurrence").gte(cfg.PERMANENT_WATER_OCCURRENCE).unmask(0)
ocean_no_dem = dem.mask().Not().unmask(1)
permanent_water = gsw_permanent.Or(wc_water).Or(ocean_no_dem).rename("permanent_water")

# Per scene: dB -> linear, Lee filter (3x3, ENL 4.9), angular terrain flattening
# (Vollrath et al. 2020, volume model, geometric layover/shadow mask); then the
# per-pixel median, back to dB. Shared with the UI: processing/s1_preprocess.py.
from processing.s1_preprocess import composite_db, lee_filter, terrain_geometry, to_db, to_natural

tf_slope, tf_phi_s = terrain_geometry(aoi)


def build_composite(col):
    return composite_db(col, aoi, tf_slope, tf_phi_s, enl=cfg.SENTINEL1_ENL)

co_col = s1_iw_vv_vh_angle(cfg.START_DATE, cfg.END_DATE)
from processing.s1_tracks import choose_track, filter_track

# Pick ONE orbit track: it must cover >=95% of the AOI, then it is the one whose
# scene is closest to FLOOD_PEAK_DATE. Event and historical stacks both use it, so
# ΔWI compares one viewing geometry. Falls back to mixing tracks (with a warning)
# if no single track covers the AOI, or if MATCH_ORBIT = False in layer_config.py.
track = choose_track(co_col, aoi, cfg.FLOOD_PEAK_DATE, match=cfg.MATCH_ORBIT)
print(track["message"])
closest = ee.Image(co_col.filter(ee.Filter.eq("system:index", track["closest_id"])).first()).set(
    "peak_diff_days", track["peak_diff_days"])
print("Scene closest to peak on that track:", track["closest_id"], f"| |Δdays| = {track['peak_diff_days']:.2f}")


def same_track(col):
    return filter_track(col, track)


co_col = same_track(co_col)
if track["matched"]:
    # Event image = the single pass nearest the peak (all its slices), not an average
    # of every pass in the window, which would dilute the flood with drained acquisitions.
    _t_peak = closest.date()
    co_col = co_col.filterDate(_t_peak.advance(-1, "hour"), _t_peak.advance(1, "hour"))
n_co = co_col.size().getInfo()
print("Event scenes used (slices of the peak pass):", n_co)
co_db = build_composite(co_col)

from datetime import date, timedelta
hist_start = (date.fromisoformat(cfg.START_DATE) - timedelta(days=int(round(365.25 * cfg.HIST_LOOKBACK_YEARS)))).isoformat()
hist_col = same_track(s1_iw_vv_vh_angle(hist_start, cfg.START_DATE))
print("Historical S1 scenes:", hist_col.size().getInfo(), "|", hist_start, "→", cfg.START_DATE)
hist_db = build_composite(hist_col)

## 2. Rebuild `final_flood`
Same as `01_` sections 3, 3b, 4: 2-D GMM on standardised (`WI`, `ΔWI`), then WorldCover / JRC / DEM / MMU post-masks. Not Otsu or `WI_CHANGE_MIN_DB`.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

N_CLUSTERS = 4
CLUSTER_SAMPLE_PIXELS = 8000
CLUSTER_SEED = 0
_GETINFO_CHUNK = 4000

water_index = co_db.select("VV").add(co_db.select("VH")).rename("WI")
hist_wi = hist_db.select("VV").add(hist_db.select("VH")).rename("WI_hist")
wi_change = hist_wi.subtract(water_index).rename("WI_change")

feat = water_index.addBands(wi_change).updateMask(permanent_water.Not())
feat_stats = feat.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=SCALE, bestEffort=True, maxPixels=1e9, tileScale=4,
).getInfo()
wi_mean, wi_std = feat_stats["WI_mean"], feat_stats["WI_stdDev"]
dw_mean, dw_std = feat_stats["WI_change_mean"], feat_stats["WI_change_stdDev"]
if wi_std is None or dw_std is None or wi_std == 0 or dw_std == 0:
    raise ValueError("Could not standardise WI / ΔWI — check AOI coverage.")

wi_z = water_index.subtract(wi_mean).divide(wi_std).rename("WI_z")
dw_z = wi_change.subtract(dw_mean).divide(dw_std).rename("dWI_z")
feat_z = wi_z.addBands(dw_z).updateMask(permanent_water.Not()).clip(aoi)
training = feat_z.sample(
    region=aoi, scale=SCALE, numPixels=CLUSTER_SAMPLE_PIXELS,
    seed=CLUSTER_SEED, geometries=False, tileScale=4,
)

def _fc_props_to_array(fc, props, chunk=_GETINFO_CHUNK):
    n = int(fc.size().getInfo() or 0)
    rows = []
    for start in range(0, n, chunk):
        feats = ee.FeatureCollection(fc.toList(chunk, start)).getInfo()["features"]
        for f in feats:
            rows.append([f["properties"].get(p) for p in props])
    return np.array(rows, dtype=float)

X = _fc_props_to_array(training, ["WI_z", "dWI_z"])
X = X[np.isfinite(X).all(axis=1)]
print(f"Cluster training samples: {len(X)} (permanent water excluded)")
assert len(X) >= N_CLUSTERS * 20, "Too few samples for GMM / k-means."

gmm = GaussianMixture(
    n_components=N_CLUSTERS, covariance_type="full",
    random_state=CLUSTER_SEED, n_init=5,
).fit(X)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=CLUSTER_SEED, n_init=10).fit(X)

def _component_table(means_z, weights=None):
    wi = means_z[:, 0] * wi_std + wi_mean
    dw = means_z[:, 1] * dw_std + dw_mean
    rows = {"WI_mean_dB": wi, "dWI_mean_dB": dw, "score": dw / dw_std - means_z[:, 0]}
    if weights is not None:
        rows["weight"] = weights
    return pd.DataFrame(rows)

def _pick_flood_id(table):
    changed = table[table["dWI_mean_dB"] > 0]
    pool = changed if len(changed) else table
    return int(pool["score"].idxmax())

gmm_table = _component_table(gmm.means_, gmm.weights_)
kmeans_table = _component_table(kmeans.cluster_centers_)
gmm_flood_id = _pick_flood_id(gmm_table)
kmeans_flood_id = _pick_flood_id(kmeans_table)
print("GMM components\n", gmm_table.round(3))
print("GMM flood component:", gmm_flood_id)
print("K-means centroids\n", kmeans_table.round(3))
print("K-means flood cluster:", kmeans_flood_id)

def _gmm_loglik_image(k):
    mean, cov, weight = gmm.means_[k], gmm.covariances_[k], float(gmm.weights_[k])
    prec = np.linalg.inv(cov)
    det = float(np.linalg.det(cov))
    log_norm = np.log(max(weight, 1e-12)) - 0.5 * np.log((2.0 * np.pi) ** 2 * max(det, 1e-12))
    d0 = wi_z.subtract(float(mean[0]))
    d1 = dw_z.subtract(float(mean[1]))
    maha = (
        d0.pow(2).multiply(float(prec[0, 0]))
        .add(d0.multiply(d1).multiply(2.0 * float(prec[0, 1])))
        .add(d1.pow(2).multiply(float(prec[1, 1])))
    )
    return ee.Image.constant(log_norm).subtract(maha.multiply(0.5)).rename(f"ll_{k}")

def _kmeans_negdist_image(k):
    mean = kmeans.cluster_centers_[k]
    d0 = wi_z.subtract(float(mean[0]))
    d1 = dw_z.subtract(float(mean[1]))
    return d0.pow(2).add(d1.pow(2)).multiply(-1).rename(f"nd_{k}")

def _argmax_label(score_images):
    best = score_images[0]
    label = ee.Image.constant(0)
    for k, img in enumerate(score_images[1:], start=1):
        label = label.where(img.gt(best), k)
        best = best.max(img)
    return label.rename("cluster")

gmm_flood = _argmax_label([_gmm_loglik_image(k) for k in range(N_CLUSTERS)]).eq(gmm_flood_id).rename("gmm_flood")
kmeans_flood = _argmax_label([_kmeans_negdist_image(k) for k in range(N_CLUSTERS)]).eq(kmeans_flood_id).rename("kmeans_flood")

# WorldCover / JRC / DEM are post-masks, not GMM features (same as 01_ 3b / 4).
flood_candidate = gmm_flood.And(permanent_water.Not()).And(dark_land.Not()).rename("flood_candidate")

dem_pct = dem.reduceRegion(ee.Reducer.percentile([5, 50, 95]), aoi, 30, bestEffort=True, maxPixels=1e9).getInfo()
elev_ok = dem.lt(cfg.ELEVATION_MAX_M).unmask(1)
if cfg.ELEVATION_ABOVE_P5_M is not None:
    elev_ok = elev_ok.And(dem.lte(dem_pct["elevation_p5"] + cfg.ELEVATION_ABOVE_P5_M).unmask(1))
slope_ok = slope_deg.lt(cfg.SLOPE_MAX_DEG).unmask(1)
# Active layover/shadow from the terrain-flattening geometry: masked in the composite.
layover_shadow_risk = co_db.select("VV").mask().Not().rename("layover_shadow_risk")

flood_masked = flood_candidate.And(elev_ok).And(slope_ok).And(layover_shadow_risk.Not())
blob_size = flood_masked.selfMask().connectedPixelCount(maxSize=128, eightConnected=True)
flood_mmu = flood_masked.updateMask(blob_size.gte(cfg.MMU_PIXELS)).unmask(0)
final_flood = (flood_mmu.focal_min(radius=SCALE, units="meters")
               .focal_max(radius=SCALE, units="meters").clip(aoi).rename("flood"))

px_km2 = ee.Image.pixelArea().divide(1e6)
valid = permanent_water.Not()

def area_km2(mask_img, only_valid=True):
    m = mask_img.And(valid) if only_valid else mask_img
    v = px_km2.updateMask(m).reduceRegion(ee.Reducer.sum(), aoi, AREA_SCALE,
                                          bestEffort=True, maxPixels=1e10).get("area").getInfo()
    return 0.0 if v is None else v

print("gmm_flood km²: %.3f | kmeans_flood km²: %.3f" % (
    area_km2(gmm_flood, only_valid=False), area_km2(kmeans_flood, only_valid=False)))
print("final_flood (km², should match 01_ flood_final_km2): %.3f" % area_km2(final_flood, only_valid=False))


## 3. Reference flood extent (Dragonfly, 14 Feb)
The zip holds two layers, urban and non-urban flood. Both are kept and tagged.

In [ ]:
def fetch_zip(url, name):
    zpath = RAW_DIR / f"{name}.zip"
    if not zpath.exists():
        urlretrieve(url, zpath)
    out = RAW_DIR / name
    if not out.exists():
        zipfile.ZipFile(zpath).extractall(out)
    return out

flood_dir = fetch_zip(FLOOD_REF_URL, "dragonfly_flood_20230214")
parts = []
for name in ["non_urban_flood_nztm.gpkg", "urban_flood_nztm.gpkg"]:
    g = gpd.read_file(next(flood_dir.rglob(name))).to_crs(4326)
    g["part"] = name.replace("_nztm.gpkg", "")
    parts.append(g)
ref = gpd.clip(pd.concat(parts, ignore_index=True), cfg.AOI_BOUNDS)
ref = ref[~ref.geometry.is_empty].copy()
ref["geometry"] = ref.geometry.simplify(0.00005)
print(ref["part"].value_counts())

def to_mask(gdf):
    return ee.Image(0).paint(geemap.gdf_to_ee(gdf[["geometry"]]), 1).clip(aoi)

ref_img = to_mask(ref)

## 4. Confusion matrix and scores
Permanent water is excluded from both maps, so scores cover new inundation only.

In [ ]:
code_img = final_flood.multiply(2).add(ref_img).updateMask(valid).rename("code")  # 0 TN, 1 FN, 2 FP, 3 TP
groups = px_km2.addBands(code_img).reduceRegion(
    ee.Reducer.sum().group(groupField=1, groupName="code"),
    aoi, AREA_SCALE, bestEffort=True, maxPixels=1e10,
).get("groups").getInfo()
km2 = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
for g in groups:
    km2[int(g["code"])] = g["sum"]
TN, FN, FP, TP = km2[0], km2[1], km2[2], km2[3]

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
iou = TP / (TP + FP + FN)
N_ALL = TP + TN + FP + FN
p_obs = (TP + TN) / N_ALL
# Chance agreement from the row/column totals (area-weighted, same as pixel counts at fixed scale).
p_exp = ((TP + FP) * (TP + FN) + (FN + TN) * (FP + TN)) / N_ALL ** 2
kappa = (p_obs - p_exp) / (1 - p_exp)

cm = pd.DataFrame([[TP, FP], [FN, TN]], index=["S1: flood", "S1: dry"],
                  columns=["Reference: flood", "Reference: dry"]).round(3)
scores = pd.Series({
    "S1 flood (km²)": TP + FP, "Reference flood (km²)": TP + FN,
    "Precision": precision, "Recall": recall, "F1": f1, "IoU": iou,
    "Overall accuracy": p_obs, "Cohen's kappa": kappa,
}).round(3)

rows = []
for part in ["non_urban_flood", "urban_flood"]:
    sub = ref[ref["part"] == part]
    if len(sub) == 0:
        continue
    m = to_mask(sub)
    r_km2, hit = area_km2(m), area_km2(m.And(final_flood))
    rows.append({"reference type": part, "reference km²": r_km2, "found km²": hit,
                 "recall": hit / r_km2 if r_km2 else np.nan})
by_type = pd.DataFrame(rows).round(3)

print("Confusion matrix (km²)\n", cm, "\n")
print(scores, "\n")
print(by_type)

In [ ]:
# 4c. Method comparison: GMM vs k-means, before and after post-masks (report table)
def _post_mask(cluster_flood):
    """Same post-masks and cleaning as final_flood (section 2)."""
    m = (cluster_flood.And(permanent_water.Not()).And(dark_land.Not())
         .And(elev_ok).And(slope_ok).And(layover_shadow_risk.Not()))
    size = m.selfMask().connectedPixelCount(maxSize=128, eightConnected=True)
    m = m.updateMask(size.gte(cfg.MMU_PIXELS)).unmask(0)
    return (m.focal_min(radius=SCALE, units="meters")
             .focal_max(radius=SCALE, units="meters").clip(aoi).rename("flood"))

def _score(flood_img):
    code = flood_img.multiply(2).add(ref_img).updateMask(valid).rename("code")
    g = px_km2.addBands(code).reduceRegion(
        ee.Reducer.sum().group(groupField=1, groupName="code"),
        aoi, AREA_SCALE, bestEffort=True, maxPixels=1e10,
    ).get("groups").getInfo()
    a = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
    for row in g:
        a[int(row["code"])] = row["sum"]
    tn, fn, fp, tp = a[0], a[1], a[2], a[3]
    n = tn + fn + fp + tp
    p = tp / (tp + fp) if tp + fp else np.nan
    r = tp / (tp + fn) if tp + fn else np.nan
    po = (tp + tn) / n
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n ** 2
    urban = ref[ref["part"] == "urban_flood"]
    u_ref = area_km2(to_mask(urban)) if len(urban) else np.nan
    u_hit = area_km2(to_mask(urban).And(flood_img)) if len(urban) else np.nan
    return {
        "Flood mapped (km²)": tp + fp,
        "Precision": p, "Recall": r,
        "F1": 2 * p * r / (p + r), "IoU": tp / (tp + fp + fn),
        "Cohen's kappa": (po - pe) / (1 - pe), "Overall accuracy": po,
        "Urban recall": u_hit / u_ref if u_ref else np.nan,
    }

methods = {
    "GMM, cluster only": gmm_flood,
    "GMM + post-masks (final)": final_flood,
    "k-means, cluster only": kmeans_flood,
    "k-means + post-masks": _post_mask(kmeans_flood),
}
method_scores = pd.DataFrame({name: _score(img) for name, img in methods.items()}).T.round(3)
method_scores.index.name = "Method"
method_scores.to_csv(OUT_DIR / "method_comparison.csv")
display(method_scores)

## 4d. Otsu threshold baseline

In [ ]:
# 4d. Non-learning baseline: Otsu thresholds on WI and ΔWI, then the same post-masks
def _otsu(hist):
    """Otsu (1979) threshold from an Earth Engine histogram (maximises between-class variance)."""
    counts = np.array(hist["histogram"], float)
    centres = np.array(hist["bucketMeans"], float)
    w0 = np.cumsum(counts); w1 = counts.sum() - w0
    m0 = np.cumsum(counts * centres); mt = m0[-1]
    mu0 = m0 / np.where(w0 > 0, w0, 1); mu1 = (mt - m0) / np.where(w1 > 0, w1, 1)
    return float(centres[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)])

_feat = water_index.addBands(wi_change).updateMask(permanent_water.Not())
_hist = _feat.reduceRegion(ee.Reducer.histogram(maxBuckets=256), aoi, SCALE,
                           bestEffort=True, maxPixels=1e9, tileScale=4).getInfo()
t_wi, t_dwi = _otsu(_hist["WI"]), _otsu(_hist["WI_change"])
print(f"Otsu thresholds: flood if WI < {t_wi:.1f} dB and ΔWI > {t_dwi:.1f} dB")

otsu_flood = water_index.lt(t_wi).And(wi_change.gt(t_dwi)).rename("otsu_flood")
otsu_row = pd.DataFrame({"S1 Otsu threshold baseline (no learning)": _score(_post_mask(otsu_flood))}).T
method_scores = pd.concat([method_scores.drop(index=otsu_row.index, errors="ignore"), otsu_row]).round(3)
method_scores.index.name = "Method"
method_scores.to_csv(OUT_DIR / "method_comparison.csv")
display(method_scores)


## 4g. Random forest, spatial cross-validation within this AOI

In [ ]:
# 4g. Supervised comparison: random forest (Dragonfly labels) with spatial block cross-validation
N_BLOCKS = 4          # 4 x 4 grid of ~2.5 km blocks over the 10 km AOI
RF_TREES = 200
RF_SAMPLES = 10000

# Features: event backscatter, change from the same-track baseline, and terrain
ev_vv, ev_vh = co_db.select("VV").rename("VV"), co_db.select("VH").rename("VH")
dvv = hist_db.select("VV").subtract(co_db.select("VV")).rename("dVV")
dvh = hist_db.select("VH").subtract(co_db.select("VH")).rename("dVH")
features = (ev_vv.addBands(ev_vh).addBands(water_index.rename("WI")).addBands(wi_change.rename("dWI"))
            .addBands(dvv).addBands(dvh).addBands(dem.rename("elevation")).addBands(slope_deg.rename("slope"))
            .updateMask(valid).clip(aoi))
BANDS = features.bandNames().getInfo()

# Fold id: blocks in a checkerboard, so each fold's 4 blocks are spread across the AOI
W, S, E, N = cfg.AOI_BOUNDS
_ll = ee.Image.pixelLonLat()
_col = _ll.select("longitude").subtract(W).divide((E - W) / N_BLOCKS).floor().clamp(0, N_BLOCKS - 1)
_row = _ll.select("latitude").subtract(S).divide((N - S) / N_BLOCKS).floor().clamp(0, N_BLOCKS - 1)
fold_img = _row.mod(2).multiply(2).add(_col.mod(2)).toInt().rename("fold")

samples = features.addBands(ref_img.rename("label").toInt()).addBands(fold_img).sample(
    region=aoi, scale=SCALE, numPixels=RF_SAMPLES, seed=CLUSTER_SEED, geometries=False, tileScale=4)
print(f"Labelled samples: {samples.size().getInfo()}  (label counts {samples.aggregate_histogram('label').getInfo()})")

preds, importances, oob = [], [], []
for f in range(4):
    train = samples.filter(ee.Filter.neq("fold", f))
    clf = ee.Classifier.smileRandomForest(numberOfTrees=RF_TREES, minLeafPopulation=5, seed=f).train(
        features=train, classProperty="label", inputProperties=BANDS)
    preds.append(features.classify(clf).updateMask(fold_img.eq(f)))      # predict only the held-out blocks
    exp = clf.explain().getInfo()
    importances.append(exp["importance"]); oob.append(exp.get("outOfBagErrorEstimate", np.nan))
    print(f"fold {f}: {train.size().getInfo()} training samples, out-of-bag error {oob[-1]:.3f}")

# Every pixel predicted by a model that never saw its block
rf_raw = ee.ImageCollection(preds).mosaic().unmask(0).clip(aoi).eq(1).rename("rf_flood")
rf_rows = pd.DataFrame({"S1 random forest, spatial CV (supervised)": _score(rf_raw),
                        "S1 random forest + post-masks": _score(_post_mask(rf_raw))}).T
method_scores = pd.concat([method_scores.drop(index=rf_rows.index, errors="ignore"), rf_rows]).round(3)
method_scores.index.name = "Method"
method_scores.to_csv(OUT_DIR / "method_comparison.csv")
display(method_scores)

# Confusion matrix (km²) for the RF + post-masks map, same layout as confusion_km2.csv
_code = _post_mask(rf_raw).multiply(2).add(ref_img).updateMask(valid).rename("code")
_g = px_km2.addBands(_code).reduceRegion(ee.Reducer.sum().group(groupField=1, groupName="code"),
                                         aoi, AREA_SCALE, bestEffort=True, maxPixels=1e10).get("groups").getInfo()
_a = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
for _r in _g:
    _a[int(_r["code"])] = _r["sum"]
rf_cm = pd.DataFrame([[_a[3], _a[2]], [_a[1], _a[0]]], index=["RF: flood", "RF: dry"],
                     columns=["Reference: flood", "Reference: dry"]).round(3)
rf_cm.to_csv(OUT_DIR / "rf_confusion_km2.csv")
display(rf_cm)

# Feature importance, % of total per fold, then mean and SD across folds
imp = pd.DataFrame(importances).fillna(0)
imp = imp.div(imp.sum(axis=1), axis=0) * 100
rf_importance = pd.DataFrame({"mean %": imp.mean(), "sd %": imp.std()}).sort_values("mean %", ascending=False).round(1)
rf_importance.to_csv(OUT_DIR / "rf_feature_importance.csv")
display(rf_importance)
print(f"Mean out-of-bag error across folds: {np.nanmean(oob):.3f}")


## 4h. Transfer: random forest trained on the main AOI

In [ ]:
# 4h. Transfer test: the random forest trained on the MAIN AOI, applied unchanged to this AOI
train_csv = _repo / "outputs" / "validation" / "rf_training_samples_aoi1.csv"
assert train_csv.exists(), "Run the export cell in 04_validation.ipynb (main AOI) first."
tr = pd.read_csv(train_csv)
RF_BANDS = ["VV", "VH", "WI", "dWI", "dVV", "dVH", "elevation", "slope"]
train_fc = ee.FeatureCollection([
    ee.Feature(None, {**{b: float(r[b]) for b in RF_BANDS}, "label": int(r["label"])}) for _, r in tr.iterrows()])
print(f"Training samples from the main AOI: {len(tr)} (flood {int(tr['label'].sum())})")

clf_main = ee.Classifier.smileRandomForest(numberOfTrees=RF_TREES, minLeafPopulation=5, seed=0).train(
    features=train_fc, classProperty="label", inputProperties=RF_BANDS)
rf_transfer = features.select(RF_BANDS).classify(clf_main).unmask(0).clip(aoi).eq(1).rename("rf_transfer")

tf_rows = pd.DataFrame({"S1 random forest, trained on main AOI (transfer)": _score(rf_transfer),
                        "S1 random forest, transfer + post-masks": _score(_post_mask(rf_transfer))}).T
method_scores = pd.concat([method_scores.drop(index=tf_rows.index, errors="ignore"), tf_rows]).round(3)
method_scores.index.name = "Method"
method_scores.to_csv(OUT_DIR / "method_comparison.csv")
display(method_scores)

ref_km2 = area_km2(ref_img.eq(1))
print(f"Second AOI {cfg.MAP_CENTER}: reference flood {ref_km2:.2f} km² (permanent water excluded)")
print("Results saved to", OUT_DIR)


## 5. Agreement map

In [ ]:
agree = code_img.remap([1, 2, 3], [1, 2, 3]).selfMask()
AGREE_VIS = {"min": 1, "max": 3, "palette": ["#2c7bb6", "#d7191c", "#1a9641"]}

Map = geemap.Map(center=cfg.MAP_CENTER, zoom=12, basemap="HYBRID")
Map.addLayer(agree, AGREE_VIS, "Agreement (blue = reference only, red = S1 only, green = both)")
Map

In [ ]:
%matplotlib inline
from matplotlib.ticker import FuncFormatter

W, S, E, N = cfg.AOI_BOUNDS
lat0 = (S + N) / 2
km_per_deg_lon = 111.32 * np.cos(np.radians(lat0))

# Background = event Sentinel-1 VV (grey); permanent water pale blue; agreement classes on top.
bg = co_db.select("VV").focal_median(1, "square", "pixels").visualize(min=-22, max=0)  # 3x3 median hides the frame seam (display only)
pw = permanent_water.selfMask().visualize(palette=["#c6dbef"])
rgb = bg.blend(pw).blend(agree.visualize(**AGREE_VIS))
from urllib.error import HTTPError


def ee_png(img, region, sizes=(1200, 900, 600)):
    """Fetch an EE thumbnail, stepping down in size if EE refuses (time/memory limits).

    ~1000 px across a 10 km AOI is about the 10 m analysis resolution; larger
    sizes recompute the whole chain finer than the analysis and can time out.
    """
    for dims in sizes:
        url = img.getThumbURL({"region": region, "dimensions": dims, "format": "png", "crs": "EPSG:4326"})
        try:
            return np.array(Image.open(io.BytesIO(urlopen(url).read())))
        except HTTPError as err:
            print(f"Thumbnail at {dims}px refused ({err.code}): {err.read().decode(errors='ignore')[:300]}")
    raise RuntimeError("Earth Engine refused every thumbnail size; see the messages above.")


arr = ee_png(rgb, aoi)

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(arr, extent=[W, E, S, N], interpolation="nearest")
ax.set_aspect(1 / np.cos(np.radians(lat0)))   # equal ground distance in x and y
ax.set_xlim(W, E); ax.set_ylim(S, N)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.2f}°E"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{abs(v):.2f}°S"))
ax.tick_params(labelsize=8)
for lab in ax.get_yticklabels():
    lab.set_rotation(90); lab.set_va("center")

# ESRI-style alternating scale bar + north arrow (shared with the app figure).
from processing.map_furniture import add_north_arrow, add_scale_bar
add_scale_bar(ax, km_per_deg_lon)
add_north_arrow(ax, loc="upper left")

ax.legend(handles=[Patch(color="#1a9641", label="Both flood (TP)"),
                   Patch(color="#d7191c", label="Sentinel-1 only (FP)"),
                   Patch(color="#2c7bb6", label="Reference only (FN)"),
                   Patch(facecolor="#c6dbef", edgecolor="0.4", label="Permanent water (excluded)")],
          loc="lower right", fontsize=8, framealpha=0.92)
fig.savefig(OUT_DIR / "agreement.png", dpi=300, bbox_inches="tight")
plt.show()

# Paste this as the caption in the report (not drawn on the figure).
print(
    f"Agreement between the Sentinel-1 GMM flood map ({cfg.START_DATE} to {cfg.END_DATE}) and the "
    f"Dragonfly Data Science 14 February 2023 flood extent (CC BY-SA 3.0 IGO), Cyclone Gabrielle, Hawke's Bay. "
    f"AOI ±{cfg.AOI_RADIUS_KM:g} km around {cfg.MAP_CENTER[0]:.2f}°, {cfg.MAP_CENTER[1]:.2f}°. "
    f"Background: event Sentinel-1 VV backscatter, 3×3 median for display (Copernicus Sentinel data 2023). WGS 84 (EPSG:4326). "
    f"IoU = {iou:.2f}, F1 = {f1:.2f}, kappa = {kappa:.2f}. Permanent water excluded from scoring."
)